# LeetCode 2956 — Find Common Elements Between Two Arrays bằng Python

Link đề: https://leetcode.com/problems/find-common-elements-between-two-arrays/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findIntersectionValues(self, nums1: List[int], nums2: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm **số vị trí** (không phải số giá trị khác nhau!) trong mỗi mảng mà phần tử ở đó **có xuất hiện trong mảng còn lại**, và trả về cặp `[answer1, answer2]`.
2. Biết dùng **tập hợp (`set`)** để kiểm tra "có xuất hiện không" trong thời gian trung bình `O(1)`, và hiểu vì sao `x in list` chậm hơn `x in set`.
3. Biết **mảng đánh dấu / mảng đếm** theo giá trị (vì giá trị chỉ từ `1` đến `100`) và `collections.Counter`.
4. Đi từ cách **duyệt lồng nhau bằng `in` trên list** → **chuyển mảng kia thành `set`** → **mảng đánh dấu `seen[v]`** → **`Counter` và cộng tần suất**.
5. Tránh các lỗi kinh điển: đếm **giá trị khác nhau** thay vì **số vị trí** (bỏ qua phần tử lặp), chỉ tính một chiều, nhầm `nums1 in nums2`, và vừa duyệt vừa xóa phần tử trong list.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` (độ dài `n`) và `nums2` (độ dài `m`), đánh chỉ số từ `0`. Hãy tính:

- `answer1`: **số chỉ số `i`** sao cho `nums1[i]` **xuất hiện ít nhất một lần** trong `nums2`;
- `answer2`: **số chỉ số `j`** sao cho `nums2[j]` **xuất hiện ít nhất một lần** trong `nums1`.

Trả về mảng `[answer1, answer2]`.

**Ví dụ 1**

- Input: `nums1 = [4, 3, 2, 3, 1]`, `nums2 = [2, 2, 5, 2, 3, 6]`
- Output: `[3, 4]`
- Giải thích: trong `nums1`, các phần tử ở chỉ số `1, 2, 3` (giá trị `3, 2, 3`) có trong `nums2`, nên `answer1 = 3`. Trong `nums2`, các phần tử ở chỉ số `0, 1, 3, 4` (giá trị `2, 2, 2, 3`) có trong `nums1`, nên `answer2 = 4`.

**Ví dụ 2**

- Input: `nums1 = [3, 4, 2, 3]`, `nums2 = [1, 5]`
- Output: `[0, 0]`
- Giải thích: hai mảng không có phần tử chung.

**Ràng buộc:** `n == nums1.length`; `m == nums2.length`; `1 <= n, m <= 100`; `1 <= nums1[i], nums2[i] <= 100`.

In [1]:
from typing import List
from collections import Counter
from bisect import bisect_left
import random

examples = [
    ([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6], [3, 4]),
    ([3, 4, 2, 3], [1, 5], [0, 0]),
    ([2, 3, 2], [1, 2], [2, 1]),
]
for nums1, nums2, expected in examples:
    print("nums1 =", nums1, "nums2 =", nums2, "-> kỳ vọng", expected)

nums1 = [4, 3, 2, 3, 1] nums2 = [2, 2, 5, 2, 3, 6] -> kỳ vọng [3, 4]
nums1 = [3, 4, 2, 3] nums2 = [1, 5] -> kỳ vọng [0, 0]
nums1 = [2, 3, 2] nums2 = [1, 2] -> kỳ vọng [2, 1]


## 3. Kiến thức nền

### 3.1. Toán tử `in` với list

`x in a` với `a` là list sẽ **quét từng phần tử** của `a` từ đầu đến khi gặp `x` (hoặc hết). Trong tệ nhất tốn **`O(len(a))`**. Nó trả về `True/False` chứ không đếm số lần.

In [2]:
a = [2, 2, 5, 2, 3, 6]
print(3 in a, 7 in a)                    # True False
print(2 in a)                            # True (dù 2 xuất hiện 3 lần, kết quả chỉ là True)

True False
True


### 3.2. Tập hợp `set`

`set` là tập các giá trị **khác nhau**, **không có thứ tự**. Tạo bằng `set(a)`; kiểm tra thuộc `x in s` chạy trung bình **`O(1)`** nhờ bảng băm (*hash table*). Việc `set(a)` cũng loại các phần tử trùng.

In [3]:
a = [2, 2, 5, 2, 3, 6]
s = set(a)
print(s, len(s))                         # {2, 3, 5, 6} 4
print(3 in s, 7 in s)                    # True False
print(len(a), len(s))                    # 6 4: số phần tử vs số giá trị khác nhau

{2, 3, 5, 6} 4
True False
6 4


### 3.3. `True` cộng được như số `1`

Trong Python `True == 1` và `False == 0`. Vì vậy `sum(điều_kiện for ...)` **đếm số lần điều kiện đúng**. Đây là cách đếm gọn nhất.

In [4]:
print(True + True + False)               # 2
nums1, s2 = [4, 3, 2, 3, 1], {2, 3, 5, 6}
print(sum(x in s2 for x in nums1))       # 3: số vị trí của nums1 có giá trị nằm trong s2

2
3


### 3.4. Mảng đánh dấu theo giá trị

Vì giá trị chỉ từ `1` đến `100`, ta có thể dùng **mảng 101 phần tử** `seen`, trong đó `seen[v] = True` nghĩa là "giá trị `v` có xuất hiện". Tra cứu `seen[v]` là truy cập mảng: `O(1)` chắc chắn (không "trung bình").

In [5]:
seen = [False] * 101
for v in [2, 2, 5, 2, 3, 6]:
    seen[v] = True
print([v for v in range(101) if seen[v]])   # [2, 3, 5, 6]
print(seen[3], seen[4])                      # True False

[2, 3, 5, 6]
True False


### 3.5. `collections.Counter`

`Counter(a)` tạo dict `giá_trị -> số_lần`. Tra một giá trị không có sẽ ra `0` (không báo lỗi). Rất hợp với bài "đếm vị trí" vì **số vị trí có giá trị `v` chính là `Counter[v]`**.

In [6]:
c = Counter([4, 3, 2, 3, 1])
print(c)                                 # Counter({3: 2, 4: 1, 2: 1, 1: 1})
print(c[3], c[9])                        # 2 0
print(sum(c.values()))                   # 5 = len(a)

Counter({3: 2, 4: 1, 2: 1, 1: 1})
2 0
5


### 3.6. Sắp xếp và tìm nhị phân (`bisect_left`)

`bisect_left(b, x)` trên danh sách **đã sắp xếp** `b` trả về vị trí chèn `x`; nếu `b[pos] == x` thì `x` có trong `b`. Mất `O(log m)` mỗi lần tra. Ta chỉ dùng để so sánh với các cách khác.

In [7]:
b = sorted([2, 2, 5, 2, 3, 6])           # [2, 2, 2, 3, 5, 6]
for x in (3, 4):
    pos = bisect_left(b, x)
    print(x, "->", pos, "có" if pos < len(b) and b[pos] == x else "không có")

3 -> 3 có
4 -> 4 không có


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với mỗi phần tử của `nums1` hỏi "giá trị này có nằm trong `nums2` không?", đếm số câu trả lời "có"; rồi làm ngược lại. Tính tay Ví dụ 1:

| `nums1[i]` | có trong `nums2 = [2,2,5,2,3,6]`? |
|---|---|
| 4 | không |
| 3 | **có** |
| 2 | **có** |
| 3 | **có** |
| 1 | không |

`answer1 = 3`. Ngược lại:

| `nums2[j]` | có trong `nums1 = [4,3,2,3,1]`? |
|---|---|
| 2 | **có** |
| 2 | **có** |
| 5 | không |
| 2 | **có** |
| 3 | **có** |
| 6 | không |

`answer2 = 4`. **Chú ý:** `3` xuất hiện hai lần trong `nums1` nên đóng góp **hai** vị trí, và `2` xuất hiện ba lần trong `nums2` nên đóng góp **ba** vị trí. Đề đếm **vị trí**, không đếm giá trị khác nhau.

In [8]:
nums1, nums2 = [4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]
print("answer1:", [(x, x in nums2) for x in nums1])
print("answer2:", [(y, y in nums1) for y in nums2])
print("đếm vị trí:", sum(x in nums2 for x in nums1), sum(y in nums1 for y in nums2))
print("số giá trị chung khác nhau:", len(set(nums1) & set(nums2)))     # 2, KHÁC với 3 và 4

answer1: [(4, False), (3, True), (2, True), (3, True), (1, False)]
answer2: [(2, True), (2, True), (5, False), (2, True), (3, True), (6, False)]
đếm vị trí: 3 4
số giá trị chung khác nhau: 2


## 5. Cách 1 — Brute Force / baseline: `in` trên list, hai vòng đếm

**Ý tưởng.** Hai vòng riêng biệt. Vòng 1: với mỗi `x` trong `nums1`, nếu `x in nums2` (quét list) thì `a1 += 1`. Vòng 2: ngược lại. Không dùng cấu trúc dữ liệu đặc biệt.

In [9]:
def find_intersection_brute(nums1: List[int], nums2: List[int]) -> List[int]:
    a1 = 0
    for x in nums1:
        if x in nums2:                      # quét list nums2: O(m)
            a1 += 1
    a2 = 0
    for y in nums2:
        if y in nums1:                      # quét list nums1: O(n)
            a2 += 1
    return [a1, a2]


for nums1, nums2, expected in examples:
    print(nums1, nums2, "->", find_intersection_brute(nums1, nums2), "(kỳ vọng", expected, ")")

[4, 3, 2, 3, 1] [2, 2, 5, 2, 3, 6] -> [3, 4] (kỳ vọng [3, 4] )
[3, 4, 2, 3] [1, 5] -> [0, 0] (kỳ vọng [0, 0] )
[2, 3, 2] [1, 2] -> [2, 1] (kỳ vọng [2, 1] )


In [10]:
# Xem hai vòng đếm làm việc (ví dụ 1): in ra từng phép kiểm tra
nums1, nums2 = [4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]
a1 = 0
for i, x in enumerate(nums1):
    ok = x in nums2
    a1 += ok
    print(f"nums1[{i}]={x}: có trong nums2? {ok!s:5} -> a1={a1}")

nums1[0]=4: có trong nums2? False -> a1=0
nums1[1]=3: có trong nums2? True  -> a1=1
nums1[2]=2: có trong nums2? True  -> a1=2
nums1[3]=3: có trong nums2? True  -> a1=3
nums1[4]=1: có trong nums2? False -> a1=3


**Độ phức tạp (Cách 1).** `x in list` tốn `O(m)` và ta làm `n` lần, rồi ngược lại: **Time `O(n · m)`**, **Space `O(1)`**.

**Nút thắt.** Mỗi câu hỏi "có trong mảng kia không?" lại **quét lại cả mảng**. Ta chỉ cần xây một cấu trúc để trả lời câu hỏi đó nhanh hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Câu hỏi "giá trị `v` có trong mảng kia không?" chỉ phụ thuộc **tập các giá trị** của mảng kia, không phụ thuộc thứ tự hay số lần xuất hiện. Vậy:

1. Dựng **một lần** tập `s1 = set(nums1)` và `s2 = set(nums2)` (mỗi mảng `O(n)`).
2. Mỗi câu hỏi "`x` có trong mảng kia?" thành `x in s2` / `y in s1`: **`O(1)`**.
3. Vẫn **đếm theo từng vị trí** (duyệt mảng gốc, có lặp), nên `3` xuất hiện hai lần được đếm hai lần.

Tổng thời gian giảm từ `O(n·m)` xuống `O(n + m)`. Vì giá trị bị chặn `<= 100`, mảng đánh dấu hay `Counter` cũng làm được tương tự.

## 7. Các cách tiếp theo

### Cách 2 — Chuyển mỗi mảng thành `set`, rồi `sum(x in set ...)`

**Giải thích.** `s1 = set(nums1)`, `s2 = set(nums2)`. `answer1 = sum(x in s2 for x in nums1)` (duyệt **mảng gốc** để đếm đủ phần tử lặp), `answer2 = sum(y in s1 for y in nums2)`.

In [11]:
def find_intersection_set(nums1: List[int], nums2: List[int]) -> List[int]:
    s1, s2 = set(nums1), set(nums2)
    a1 = sum(x in s2 for x in nums1)         # duyệt nums1 (mảng gốc) -> đếm cả phần tử lặp
    a2 = sum(y in s1 for y in nums2)
    return [a1, a2]


for nums1, nums2, expected in examples:
    print(nums1, nums2, "->", find_intersection_set(nums1, nums2), "(kỳ vọng", expected, ")")

[4, 3, 2, 3, 1] [2, 2, 5, 2, 3, 6] -> [3, 4] (kỳ vọng [3, 4] )
[3, 4, 2, 3] [1, 5] -> [0, 0] (kỳ vọng [0, 0] )
[2, 3, 2] [1, 2] -> [2, 1] (kỳ vọng [2, 1] )


**Độ phức tạp Cách 2:** **Time `O(n + m)`** (trung bình), **Space `O(n + m)`** cho hai tập.

### Cách 3 — Mảng đánh dấu theo giá trị (vì `1 <= v <= 100`)

**Giải thích.** Dùng hai mảng boolean `in1[v]`, `in2[v]` độ dài `101`. Duyệt `nums1` đánh dấu `in1[x] = True`; duyệt `nums2` đánh dấu `in2[y] = True`. Rồi `answer1 = số x trong nums1 mà in2[x]`, `answer2 = số y trong nums2 mà in1[y]`. Mỗi lần tra là truy cập chỉ số `O(1)` thật sự, không phụ thuộc hàm băm.

In [12]:
def find_intersection_mark(nums1: List[int], nums2: List[int]) -> List[int]:
    in1 = [False] * 101
    in2 = [False] * 101
    for x in nums1:
        in1[x] = True
    for y in nums2:
        in2[y] = True
    a1 = 0
    for x in nums1:
        if in2[x]:
            a1 += 1
    a2 = 0
    for y in nums2:
        if in1[y]:
            a2 += 1
    return [a1, a2]


for nums1, nums2, expected in examples:
    print(nums1, nums2, "->", find_intersection_mark(nums1, nums2), "(kỳ vọng", expected, ")")

[4, 3, 2, 3, 1] [2, 2, 5, 2, 3, 6] -> [3, 4] (kỳ vọng [3, 4] )
[3, 4, 2, 3] [1, 5] -> [0, 0] (kỳ vọng [0, 0] )
[2, 3, 2] [1, 2] -> [2, 1] (kỳ vọng [2, 1] )


In [13]:
# Quan sát mảng đánh dấu của ví dụ 1: chỉ in các giá trị có mặt
nums1, nums2 = [4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]
in1, in2 = [False] * 101, [False] * 101
for x in nums1:
    in1[x] = True
for y in nums2:
    in2[y] = True
print("giá trị có trong nums1:", [v for v in range(101) if in1[v]])
print("giá trị có trong nums2:", [v for v in range(101) if in2[v]])
print("giá trị chung:", [v for v in range(101) if in1[v] and in2[v]])

giá trị có trong nums1: [1, 2, 3, 4]
giá trị có trong nums2: [2, 3, 5, 6]
giá trị chung: [2, 3]


**Độ phức tạp Cách 3:** **Time `O(n + m + V)`** với `V = 101` (khởi tạo mảng), **Space `O(V)`** (hằng số vì `V` cố định).

### Cách 4 — `Counter` và cộng tần suất của các giá trị chung

**Giải thích.** `c1 = Counter(nums1)`, `c2 = Counter(nums2)`. Số vị trí của `nums1` có giá trị nằm trong `nums2` bằng **tổng tần suất `c1[v]` của mọi giá trị `v` chung**. Nói cách khác `answer1 = sum(c1[v] for v in c1 if v in c2)` và `answer2 = sum(c2[v] for v in c2 if v in c1)`. Duyệt theo **giá trị khác nhau** thay vì từng vị trí.

In [14]:
def find_intersection_counter(nums1: List[int], nums2: List[int]) -> List[int]:
    c1, c2 = Counter(nums1), Counter(nums2)
    a1 = sum(cnt for v, cnt in c1.items() if v in c2)
    a2 = sum(cnt for v, cnt in c2.items() if v in c1)
    return [a1, a2]


for nums1, nums2, expected in examples:
    print(nums1, nums2, "->", find_intersection_counter(nums1, nums2), "(kỳ vọng", expected, ")")

# Quan sát: các giá trị chung và tần suất trong từng mảng
c1, c2 = Counter([4, 3, 2, 3, 1]), Counter([2, 2, 5, 2, 3, 6])
common = c1.keys() & c2.keys()
print("giá trị chung:", sorted(common))
print("tần suất trong nums1:", {v: c1[v] for v in sorted(common)})     # {2: 1, 3: 2} -> tổng 3
print("tần suất trong nums2:", {v: c2[v] for v in sorted(common)})     # {2: 3, 3: 1} -> tổng 4

[4, 3, 2, 3, 1] [2, 2, 5, 2, 3, 6] -> [3, 4] (kỳ vọng [3, 4] )
[3, 4, 2, 3] [1, 5] -> [0, 0] (kỳ vọng [0, 0] )
[2, 3, 2] [1, 2] -> [2, 1] (kỳ vọng [2, 1] )
giá trị chung: [2, 3]
tần suất trong nums1: {2: 1, 3: 2}
tần suất trong nums2: {2: 3, 3: 1}


**Độ phức tạp Cách 4:** **Time `O(n + m)`**, **Space `O(n + m)`** (hai `Counter`). Tiện nếu còn cần biết "mỗi giá trị chung góp bao nhiêu vị trí".

In [15]:
# Code nộp LeetCode cho Cách 2 (ngắn, rõ, tối ưu O(n + m))
from typing import List

class Solution:
    def findIntersectionValues(self, nums1: List[int], nums2: List[int]) -> List[int]:
        s1, s2 = set(nums1), set(nums2)
        return [sum(x in s2 for x in nums1), sum(y in s1 for y in nums2)]

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 (set) trên Ví dụ 1; (b) Cách 4 (Counter) cộng tần suất.

In [16]:
def trace_set(nums1, nums2):
    s1, s2 = set(nums1), set(nums2)
    print("s1 =", sorted(s1), "| s2 =", sorted(s2))
    a1 = 0
    for i, x in enumerate(nums1):
        hit = x in s2
        a1 += hit
        print(f"nums1[{i}]={x}: {x} in s2? {hit!s:5} -> a1={a1}")
    a2 = 0
    for j, y in enumerate(nums2):
        hit = y in s1
        a2 += hit
        print(f"nums2[{j}]={y}: {y} in s1? {hit!s:5} -> a2={a2}")
    print("=> [", a1, ",", a2, "]")
    return [a1, a2]


assert trace_set([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]) == [3, 4]

s1 = [1, 2, 3, 4] | s2 = [2, 3, 5, 6]
nums1[0]=4: 4 in s2? False -> a1=0
nums1[1]=3: 3 in s2? True  -> a1=1
nums1[2]=2: 2 in s2? True  -> a1=2
nums1[3]=3: 3 in s2? True  -> a1=3
nums1[4]=1: 1 in s2? False -> a1=3
nums2[0]=2: 2 in s1? True  -> a2=1
nums2[1]=2: 2 in s1? True  -> a2=2
nums2[2]=5: 5 in s1? False -> a2=2
nums2[3]=2: 2 in s1? True  -> a2=3
nums2[4]=3: 3 in s1? True  -> a2=4
nums2[5]=6: 6 in s1? False -> a2=4
=> [ 3 , 4 ]


In [17]:
def trace_counter(nums1, nums2):
    c1, c2 = Counter(nums1), Counter(nums2)
    print("c1 =", dict(c1))
    print("c2 =", dict(c2))
    a1 = a2 = 0
    for v, cnt in c1.items():
        if v in c2:
            a1 += cnt
            print(f"giá trị {v}: có trong cả hai -> cộng c1[{v}]={cnt} vào a1 (a1={a1})")
        else:
            print(f"giá trị {v}: chỉ có trong nums1 -> bỏ qua")
    for v, cnt in c2.items():
        if v in c1:
            a2 += cnt
            print(f"giá trị {v}: có trong cả hai -> cộng c2[{v}]={cnt} vào a2 (a2={a2})")
    print("=> [", a1, ",", a2, "]")
    return [a1, a2]


assert trace_counter([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]) == [3, 4]

c1 = {4: 1, 3: 2, 2: 1, 1: 1}
c2 = {2: 3, 5: 1, 3: 1, 6: 1}
giá trị 4: chỉ có trong nums1 -> bỏ qua
giá trị 3: có trong cả hai -> cộng c1[3]=2 vào a1 (a1=2)
giá trị 2: có trong cả hai -> cộng c1[2]=1 vào a1 (a1=3)
giá trị 1: chỉ có trong nums1 -> bỏ qua
giá trị 2: có trong cả hai -> cộng c2[2]=3 vào a2 (a2=3)
giá trị 3: có trong cả hai -> cộng c2[3]=1 vào a2 (a2=4)
=> [ 3 , 4 ]


Bảng tóm tắt (Ví dụ 1):

| giá trị chung | tần suất trong `nums1` | tần suất trong `nums2` |
|---|---|---|
| 2 | 1 | 3 |
| 3 | 2 | 1 |
| **tổng** | **3** (= `answer1`) | **4** (= `answer2`) |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (hai `set`). Đúng class `Solution`, đúng method `findIntersectionValues(self, nums1: List[int], nums2: List[int]) -> List[int]`.

In [18]:
from typing import List

class Solution:
    def findIntersectionValues(self, nums1: List[int], nums2: List[int]) -> List[int]:
        s1, s2 = set(nums1), set(nums2)           # tập giá trị khác nhau của mỗi mảng
        answer1 = sum(x in s2 for x in nums1)     # đếm VỊ TRÍ (duyệt mảng gốc, có lặp)
        answer2 = sum(y in s1 for y in nums2)
        return [answer1, answer2]


sol = Solution()
for nums1, nums2, expected in examples:
    print(nums1, nums2, "->", sol.findIntersectionValues(nums1, nums2))

[4, 3, 2, 3, 1] [2, 2, 5, 2, 3, 6] -> [3, 4]
[3, 4, 2, 3] [1, 5] -> [0, 0]
[2, 3, 2] [1, 2] -> [2, 1]


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa.** `answer1` là số chỉ số `i` mà `nums1[i]` xuất hiện trong `nums2`. Phần tử `nums1[i]` xuất hiện trong `nums2` ⇔ giá trị đó thuộc `set(nums2)`. Ta cộng `1` cho **mỗi** chỉ số thỏa, nên phần tử lặp được tính nhiều lần, đúng đề.
- **Hai chiều độc lập.** `answer1` chỉ phụ thuộc `nums1` (các vị trí) và **tập** giá trị của `nums2`; `answer2` đối xứng. Hai phép đếm không ảnh hưởng nhau.
- **`set` chỉ loại trùng ở mảng "đối chứng".** Ta dùng `set` để **tra cứu**, còn **đếm** thì duyệt mảng gốc. Nếu lỡ duyệt `set(nums1)` thay vì `nums1` thì mới mất phần tử lặp (lỗi thường gặp).
- **Cách 4 tương đương.** `Counter` gom các vị trí cùng giá trị: số vị trí có giá trị `v` là `c1[v]`; cộng `c1[v]` qua mọi `v` chung chính là số vị trí thỏa.
- **Không phụ thuộc thứ tự**; mảng không có phần tử chung cho `[0, 0]`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums1)`, `m = len(nums2)`, `V = 101` là số giá trị có thể.

- **Cách 1 (`in` trên list):** Time `O(n · m)`, Space `O(1)`.
- **Cách 2 (`set`):** Time `O(n + m)` (trung bình), Space `O(n + m)`.
- **Cách 3 (mảng đánh dấu):** Time `O(n + m + V)`, Space `O(V)` (hằng số).
- **Cách 4 (`Counter`):** Time `O(n + m)`, Space `O(n + m)`.
- Phải đọc mọi phần tử nên `O(n + m)` là tối ưu. Ở đây `n, m <= 100`: Cách 1 tối đa `2 · 10⁴` phép so sánh, vẫn nhanh, nhưng Cách 2/3 là chuẩn mực khi dữ liệu lớn.

## 12. Bộ test

In [19]:
sol = Solution()
impls = {
    "brute": find_intersection_brute,
    "set": find_intersection_set,
    "mark": find_intersection_mark,
    "counter": find_intersection_counter,
    "submit": sol.findIntersectionValues,
}

for nums1, nums2, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums1), list(nums2))
        assert got == expected, (key, nums1, nums2, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums1, nums2, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums1), list(nums2))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1], [1], [1, 1], "mỗi mảng một phần tử, bằng nhau")
check_all([1], [2], [0, 0], "mỗi mảng một phần tử, khác nhau")
check_all([5, 5, 5], [5], [3, 1], "nhiều lần ở một mảng, một lần ở mảng kia")
check_all([1, 2, 3], [3, 2, 1], [3, 3], "cùng tập giá trị, khác thứ tự")
check_all([1, 1, 1, 1], [1, 1], [4, 2], "cùng một giá trị lặp nhiều lần")
check_all([100, 1], [1, 100, 100, 100], [2, 4], "giá trị biên 1 và 100")
check_all([1, 2, 3, 4, 5], [6, 7, 8], [0, 0], "hoàn toàn rời nhau")
check_all(list(range(1, 101)), list(range(1, 101)), [100, 100], "n = m = 100, giống hệt")
check_all([1] * 100, [2] * 100, [0, 0], "n = m = 100, rời nhau")
check_all([1] * 100, [1] * 100, [100, 100], "n = m = 100, cùng một giá trị")

PASS mỗi mảng một phần tử, bằng nhau -> [1, 1]
PASS mỗi mảng một phần tử, khác nhau -> [0, 0]
PASS nhiều lần ở một mảng, một lần ở mảng kia -> [3, 1]
PASS cùng tập giá trị, khác thứ tự -> [3, 3]
PASS cùng một giá trị lặp nhiều lần -> [4, 2]
PASS giá trị biên 1 và 100 -> [2, 4]
PASS hoàn toàn rời nhau -> [0, 0]
PASS n = m = 100, giống hệt -> [100, 100]
PASS n = m = 100, rời nhau -> [0, 0]
PASS n = m = 100, cùng một giá trị -> [100, 100]


### 12.2. Tính chất và phản ví dụ

In [21]:
# Tính chất 1: nếu hai mảng giống nhau thì kết quả là [n, n]
random.seed(11)
for _ in range(200):
    a = [random.randint(1, 100) for _ in range(random.randint(1, 100))]
    assert sol.findIntersectionValues(a, list(a)) == [len(a), len(a)]
print("PASS hai mảng giống nhau -> [n, n]")

# Tính chất 2: đổi chỗ hai mảng thì đổi chỗ hai đáp án
for _ in range(500):
    a = [random.randint(1, 10) for _ in range(random.randint(1, 30))]
    b = [random.randint(1, 10) for _ in range(random.randint(1, 30))]
    r1, r2 = sol.findIntersectionValues(a, b), sol.findIntersectionValues(b, a)
    assert r1 == r2[::-1]
print("PASS đổi chỗ hai mảng thì đảo kết quả")

# Tính chất 3: answer1 = 0 <=> answer2 = 0 <=> không có giá trị chung
for _ in range(500):
    a = [random.randint(1, 6) for _ in range(random.randint(1, 8))]
    b = [random.randint(1, 6) for _ in range(random.randint(1, 8))]
    r = sol.findIntersectionValues(a, b)
    assert (r[0] == 0) == (r[1] == 0) == (not set(a) & set(b))
    assert r[0] <= len(a) and r[1] <= len(b)
print("PASS answer1 = 0 <=> answer2 = 0 <=> không có giá trị chung; answer1 <= n, answer2 <= m")

# Tính chất 4: đầu vào không bị sửa
a, b = [4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]
ca, cb = list(a), list(b)
for key, fn in impls.items():
    fn(a, b)
assert a == ca and b == cb
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: đếm giá trị khác nhau (dùng giao của hai set)
def sai_giao_set(nums1, nums2):
    k = len(set(nums1) & set(nums2))
    return [k, k]

assert sai_giao_set([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]) == [2, 2] != [3, 4]
print("PASS phản ví dụ: len(set & set) cho [2, 2] thay vì [3, 4]")

# Phản ví dụ 2: duyệt set của chính mảng đó (mất phần tử lặp)
def sai_duyet_set(nums1, nums2):
    s1, s2 = set(nums1), set(nums2)
    return [sum(x in s2 for x in s1), sum(y in s1 for y in s2)]

assert sai_duyet_set([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]) == [2, 2]
print("PASS phản ví dụ: duyệt set thay vì mảng gốc làm mất phần tử lặp")

PASS hai mảng giống nhau -> [n, n]
PASS đổi chỗ hai mảng thì đảo kết quả
PASS answer1 = 0 <=> answer2 = 0 <=> không có giá trị chung; answer1 <= n, answer2 <= m
PASS đầu vào không bị sửa
PASS phản ví dụ: len(set & set) cho [2, 2] thay vì [3, 4]
PASS phản ví dụ: duyệt set thay vì mảng gốc làm mất phần tử lặp


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
random.seed(2956)
for trial in range(3000):
    n, m = random.randint(1, 100), random.randint(1, 100)
    hi = random.choice([3, 10, 50, 100])
    nums1 = [random.randint(1, hi) for _ in range(n)]
    nums2 = [random.randint(1, hi) for _ in range(m)]
    ref = find_intersection_brute(list(nums1), list(nums2))
    for key, fn in impls.items():
        assert fn(list(nums1), list(nums2)) == ref, (key, nums1, nums2)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm giá trị khác nhau thay vì số vị trí

In [23]:
def sai_dem_gia_tri(nums1, nums2):
    common = set(nums1) & set(nums2)
    return [len(common), len(common)]


print("sai  :", sai_dem_gia_tri([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]), "| đúng:", Solution().findIntersectionValues([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]))

sai  : [2, 2] | đúng: [3, 4]


**Vì sao sai:** đề đếm **chỉ số** `i` (vị trí), nên giá trị `3` xuất hiện hai lần trong `nums1` phải đóng góp `2`. `len(set1 & set2)` chỉ đếm **số giá trị chung** (`2`), cho `[2, 2]` thay vì `[3, 4]`. Phải duyệt **mảng gốc**.

### Lỗi 2: Duyệt `set` thay vì mảng gốc

In [24]:
def sai_duyet_set_2(nums1, nums2):
    s1, s2 = set(nums1), set(nums2)
    return [sum(1 for x in s1 if x in s2), sum(1 for y in s2 if y in s1)]


print("sai  :", sai_duyet_set_2([3, 3, 3], [3]), "| đúng:", Solution().findIntersectionValues([3, 3, 3], [3]))

sai  : [1, 1] | đúng: [3, 1]


**Vì sao sai:** `set([3, 3, 3])` chỉ còn `{3}`, nên đếm được `1` thay vì `3` vị trí. Dùng `set` để **tra cứu** (`x in s2`), nhưng **đếm** bằng cách duyệt mảng gốc.

### Lỗi 3: Chỉ tính một chiều

In [25]:
def sai_mot_chieu(nums1, nums2):
    s2 = set(nums2)
    a = sum(x in s2 for x in nums1)
    return [a, a]                                          # tưởng answer2 giống answer1


print("sai  :", sai_mot_chieu([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]), "| đúng:", Solution().findIntersectionValues([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]))

sai  : [3, 3] | đúng: [3, 4]


**Vì sao sai:** `answer1` đếm vị trí trong `nums1`, `answer2` đếm vị trí trong `nums2`; hai mảng có độ dài và số lần lặp khác nhau nên hai đáp án thường **khác nhau**. Phải tính riêng từng chiều.

### Lỗi 4: Vừa duyệt vừa xóa phần tử trong list

In [26]:
def sai_xoa(nums1, nums2):
    a, b = list(nums1), list(nums2)
    for x in a:                                            # vừa duyệt a vừa xóa khỏi a
        if x not in b:
            a.remove(x)
    return len(a)


print("sai  :", sai_xoa([1, 1, 4], [9]), "| đúng answer1:", Solution().findIntersectionValues([1, 1, 4], [9])[0])

sai  : 1 | đúng answer1: 0


**Vì sao sai:** xóa phần tử trong lúc `for` đang duyệt chính list đó làm **lệch chỉ số** và **bỏ qua phần tử**. Với `[1, 1, 4]` và `b = [9]`: sau khi xóa `1` đầu tiên, list thành `[1, 4]` và vòng lặp nhảy sang chỉ số `1` (là `4`), bỏ qua `1` còn lại; cuối cùng `a = [1]` (độ dài `1` thay vì `0`). Đừng sửa danh sách đang duyệt; hãy **đếm** thay vì xóa.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "phần tử này **có trong mảng kia không**" → đổi mảng kia thành **`set`** (hoặc mảng đánh dấu nếu giá trị nhỏ).
- Phân biệt: đếm **vị trí** (có lặp) vs đếm **giá trị khác nhau** (không lặp). Đề viết "số chỉ số `i`" là đếm vị trí.
- Công thức nhớ: **`sum(x in s2 for x in nums1)`**: `set` để tra, mảng gốc để đếm.
- Giá trị nhỏ (`<= 100`, `<= 10⁵`...) thì mảng đếm/đánh dấu nhanh và chắc hơn `set`.
- `a & b` cho hai `set` là giao; `Counter(a) & Counter(b)` là giao **có số lần** (min của tần suất): khác với cách tính của bài này.
- Hai chiều đối xứng: viết một hàm trợ giúp `count_in(a, s)` rồi gọi hai lần cho gọn.

## 15. Biến thể của bài

1. Trả về **các giá trị chung** (không lặp) thay vì số lượng.
2. Đề giống nhưng đếm **số giá trị khác nhau** chung (bài "Intersection of Two Arrays").
3. Giao **có số lần** (multiset): `[1, 2, 2, 1]` và `[2, 2]` cho `[2, 2]` ("Intersection of Two Arrays II").
4. Giá trị lớn (`10⁹`) nên không dùng được mảng đánh dấu: dùng `set` hoặc sắp xếp + hai con trỏ.
5. Hai mảng **đã sắp xếp**: dùng hai con trỏ để `O(n + m)` mà không cần bộ nhớ phụ.

Thử biến thể 1, 3 và 5:

In [27]:
def common_values(nums1, nums2):
    return sorted(set(nums1) & set(nums2))


def common_multiset(nums1, nums2):
    return sorted((Counter(nums1) & Counter(nums2)).elements())


def count_with_two_pointers(a, b):
    """a, b đã sắp xếp: đếm vị trí của a có giá trị nằm trong b và ngược lại."""
    i = j = 0
    c1 = c2 = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            v = a[i]
            while i < len(a) and a[i] == v:
                c1 += 1
                i += 1
            while j < len(b) and b[j] == v:
                c2 += 1
                j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return [c1, c2]


print("1.", common_values([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]))      # [2, 3]
print("3.", common_multiset([1, 2, 2, 1], [2, 2]))                   # [2, 2]
print("5.", count_with_two_pointers(sorted([4, 3, 2, 3, 1]), sorted([2, 2, 5, 2, 3, 6])))   # [3, 4]
assert common_values([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]) == [2, 3]
assert common_multiset([1, 2, 2, 1], [2, 2]) == [2, 2]
assert count_with_two_pointers([1, 2, 3, 3, 4], [2, 2, 2, 3, 5, 6]) == [3, 4]

1. [2, 3]
3. [2, 2]
5. [3, 4]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `in` trên list, hai vòng đếm | quét lại mảng kia mỗi lần | `O(n·m)` | `O(1)` | hiểu đề, dữ liệu rất nhỏ |
| 2. Hai `set` + `sum(x in s ...)` | tra cứu `O(1)` | `O(n+m)` | `O(n+m)` | **cách nộp bài**, đa dụng |
| 3. Mảng đánh dấu `[False]*101` | tra cứu theo chỉ số | `O(n+m+V)` | `O(V)` | giá trị nhỏ, cần tốc độ ổn định |
| 4. `Counter` cộng tần suất giá trị chung | đếm theo giá trị | `O(n+m)` | `O(n+m)` | cần thống kê theo giá trị |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 in [1, 5, 9]`, `len(set([3, 3, 3, 4]))`, `True + True`, `Counter([1, 1, 2])[1]`, `Solution().findIntersectionValues([1, 1], [1])`.

**Bài 2 (tính tay).** Với `nums1 = [7, 7, 8, 9]`, `nums2 = [8, 7, 10]`: lập bảng như mục 4 cho từng chiều và tính `[answer1, answer2]`.

**Bài 3 (điền TODO).** Hoàn thành `find_intersection_todo(nums1, nums2)` theo Cách 2.

In [28]:
def find_intersection_todo(nums1, nums2):
    # TODO 1: s1 = tập các giá trị của nums1, s2 = tập các giá trị của nums2
    # TODO 2: answer1 = số phần tử x trong nums1 (mảng gốc) mà x in s2
    # TODO 3: answer2 = số phần tử y trong nums2 (mảng gốc) mà y in s1
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_intersection_mark2(nums1, nums2)` bằng **mảng đánh dấu** `[False] * 101` (Cách 3) mà không nhìn code mẫu. Kiểm tra với ba ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `common_value_report(nums1, nums2)` trả về dict `giá trị chung -> (số lần trong nums1, số lần trong nums2)`. (b) Viết `intersection_with_k(nums1, nums2, k)` trả về `[answer1, answer2]` khi chỉ tính các phần tử của mảng kia xuất hiện **ít nhất `k` lần**. (c) Viết `find_intersection_sorted(nums1, nums2)` dùng **sắp xếp + `bisect_left`** (không `set`, không `Counter`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `2`, `2`, `2`, `[2, 1]`. Bài 3: `s1, s2 = set(nums1), set(nums2)`, rồi hai câu `sum(... for ... in nums1/nums2)`.
- **Mức 2 (vừa):** Bài 2: `nums1`: `7` có trong `nums2` (2 vị trí), `8` có (1), `9` không ⇒ `answer1 = 3`. `nums2`: `8` có, `7` có, `10` không ⇒ `answer2 = 2`. Kết quả `[3, 2]`. Bài 4: mảng `in1`, `in2`, đánh dấu rồi đếm.
- **Mức 3 (sâu):** Bài 5(a): `c1, c2 = Counter(...)`, `{v: (c1[v], c2[v]) for v in c1.keys() & c2.keys()}`. 5(b): `c1, c2 = Counter(nums1), Counter(nums2)` rồi `answer1 = sum(1 for x in nums1 if c2[x] >= k)`. 5(c): `b = sorted(nums2)`, với mỗi `x` dùng `pos = bisect_left(b, x)` và kiểm tra `pos < len(b) and b[pos] == x`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert (5 in [1, 5, 9]) is True and len(set([3, 3, 3, 4])) == 2 and True + True == 2
assert Counter([1, 1, 2])[1] == 2
assert Solution().findIntersectionValues([1, 1], [1]) == [2, 1]

# Bài 2
assert Solution().findIntersectionValues([7, 7, 8, 9], [8, 7, 10]) == [3, 2]

# Bài 3
def find_intersection_todo_ans(nums1, nums2):
    s1, s2 = set(nums1), set(nums2)
    answer1 = sum(x in s2 for x in nums1)
    answer2 = sum(y in s1 for y in nums2)
    return [answer1, answer2]

for nums1, nums2, expected in examples:
    assert find_intersection_todo_ans(nums1, nums2) == expected

# Bài 4
def find_intersection_mark2(nums1, nums2):
    in1, in2 = [False] * 101, [False] * 101
    for x in nums1:
        in1[x] = True
    for y in nums2:
        in2[y] = True
    return [sum(in2[x] for x in nums1), sum(in1[y] for y in nums2)]

for nums1, nums2, expected in examples:
    assert find_intersection_mark2(nums1, nums2) == expected

# Bài 5(a)
def common_value_report(nums1, nums2):
    c1, c2 = Counter(nums1), Counter(nums2)
    return {v: (c1[v], c2[v]) for v in sorted(c1.keys() & c2.keys())}

assert common_value_report([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6]) == {2: (1, 3), 3: (2, 1)}

# Bài 5(b)
def intersection_with_k(nums1, nums2, k):
    c1, c2 = Counter(nums1), Counter(nums2)
    return [sum(1 for x in nums1 if c2[x] >= k), sum(1 for y in nums2 if c1[y] >= k)]

assert intersection_with_k([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6], 1) == [3, 4]
assert intersection_with_k([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6], 2) == [1, 1]     # chiều 1: chỉ x = 2 (c2[2] = 3); chiều 2: chỉ y = 3 (c1[3] = 2)
assert intersection_with_k([4, 3, 2, 3, 1], [2, 2, 5, 2, 3, 6], 4) == [0, 0]

# Bài 5(c)
def find_intersection_sorted(nums1, nums2):
    def count(a, b_sorted):
        c = 0
        for x in a:
            pos = bisect_left(b_sorted, x)
            if pos < len(b_sorted) and b_sorted[pos] == x:
                c += 1
        return c
    return [count(nums1, sorted(nums2)), count(nums2, sorted(nums1))]

for nums1, nums2, expected in examples:
    assert find_intersection_sorted(nums1, nums2) == expected
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (set để tra, mảng gốc để đếm vị trí)

```python
from typing import List

class Solution:
    def findIntersectionValues(self, nums1: List[int], nums2: List[int]) -> List[int]:
        s1, s2 = set(nums1), set(nums2)
        return [sum(x in s2 for x in nums1), sum(y in s1 for y in nums2)]
```

**Checklist**

- [ ] Đếm **vị trí** (duyệt mảng gốc), không đếm giá trị khác nhau (`len(set & set)`)?
- [ ] `answer1` tính trên `nums1` kiểm tra với **`nums2`**, và `answer2` làm ngược lại (không nhầm hai mảng)?
- [ ] Tra cứu bằng `set`/mảng đánh dấu, không `x in list` lặp lại (nếu dữ liệu lớn)?
- [ ] Không vừa duyệt vừa xóa phần tử trong list?
- [ ] Trả về đúng `[answer1, answer2]` (list hai số, đúng thứ tự)?
- [ ] Đã thử: không có phần tử chung (`[0, 0]`), hai mảng giống nhau, phần tử lặp nhiều lần, giá trị `1` và `100`?
- [ ] Skeleton đúng `class Solution` / `def findIntersectionValues(self, nums1: List[int], nums2: List[int]) -> List[int]`?